# 03 - SQL Component: Business Queries & Database Analysis
**Project:** Amazon India Sales Analytics & AI-Powered Business Intelligence System  
**Role:** Data Science & Analytics Intern Portfolio Project  
**Author:** Manoj (for Executive Manager Ravi & Technical Interviewers)

---

### Why this notebook is crucial for your resume:
Recruiters actively seek candidates who can bridge the gap between **SQL databases**, **Python data science**, and **Business Intelligence**.
In this notebook, we:
1. Connect to the SQLite database `database/amazon_sales.db` containing our cleaned sales table.
2. Execute production-ready SQL queries using Aggregations, Subqueries, Window Functions (`RANK()`, `DENSE_RANK()`), and `CASE WHEN` statements.
3. Validate answers to all four management questions from the Sapphire IQ specification.


In [1]:
import sqlite3
import pandas as pd
import os

# Connect to the SQLite Database
db_path = "../database/amazon_sales.db"
conn = sqlite3.connect(db_path)
print("Connected to database successfully:", db_path)

# Verify table schema
schema_df = pd.read_sql("PRAGMA table_info(amazon_sales);", conn)
schema_df[['cid', 'name', 'type', 'notnull']]


Connected to database successfully: ../database/amazon_sales.db


,cid,name,type,notnull
0,0,Order_ID,TEXT,0
1,1,Order_Date,TEXT,0
2,2,Category,TEXT,0
3,3,Product,TEXT,0
4,4,Quantity,INTEGER,0
5,5,Unit_Price_INR,REAL,0
6,6,Discount_Pct,REAL,0
7,7,Total_Sales_INR,REAL,0
8,8,Profit_INR,REAL,0
9,9,Payment_Method,TEXT,0


### Query 1: Executive KPI Overview (Slides 1 & 2)
Calculates overall revenue, profit, total orders, units sold, AOV, profit margin %, and revenue lost.


In [2]:
query_1 = """
SELECT 
    COUNT(DISTINCT Order_ID) AS total_orders,
    SUM(Quantity) AS total_units_sold,
    ROUND(SUM(Total_Sales_INR), 2) AS total_sales_inr,
    ROUND(SUM(Profit_INR), 2) AS total_profit_inr,
    ROUND(SUM(Total_Sales_INR) / COUNT(DISTINCT Order_ID), 2) AS average_order_value_inr,
    ROUND((SUM(Profit_INR) / SUM(Total_Sales_INR)) * 100, 2) AS profit_margin_pct,
    ROUND(SUM(CASE WHEN Order_Status IN ('Returned', 'Cancelled') THEN Total_Sales_INR ELSE 0 END), 2) AS total_revenue_lost_inr
FROM amazon_sales;
"""
pd.read_sql(query_1, conn)


,total_orders,total_units_sold,total_sales_inr,total_profit_inr,average_order_value_inr,profit_margin_pct,total_revenue_lost_inr
0,10000,24926,1.557899e+08,33167008.49,15578.99,21.29,15648742.64


### Query 2: Monthly Sales & Profit Trend (Slide 2)
Tracks monthly financial trajectory and order volume over time.


In [3]:
query_2 = """
SELECT 
    Year_Month,
    COUNT(DISTINCT Order_ID) AS orders_count,
    ROUND(SUM(Total_Sales_INR), 2) AS monthly_sales_inr,
    ROUND(SUM(Profit_INR), 2) AS monthly_profit_inr,
    ROUND((SUM(Profit_INR) / SUM(Total_Sales_INR)) * 100, 2) AS profit_margin_pct
FROM amazon_sales
GROUP BY Year_Month
ORDER BY Year_Month ASC
LIMIT 12;
"""
pd.read_sql(query_2, conn)


,Year_Month,orders_count,monthly_sales_inr,monthly_profit_inr,profit_margin_pct
0,2024-01,316,4441265.57,1016530.29,22.89
1,2024-02,265,4071073.66,778165.74,19.11
2,2024-03,310,5761054.84,1243584.05,21.59
3,2024-04,326,4612546.48,1041978.73,22.59
4,2024-05,324,4660178.05,1020331.21,21.89
5,2024-06,311,4810938.17,937123.34,19.48
6,2024-07,328,5744760.13,1256434.89,21.87
7,2024-08,315,4760372.35,976600.73,20.52
8,2024-09,303,4033814.56,914060.89,22.66
9,2024-10,311,4820971.84,1033414.57,21.44


### Query 3: Category Performance & Ranking with Window Functions (Slide 3)
Uses SQL Window Function `DENSE_RANK()` to rank categories by revenue generation.


In [4]:
query_3 = """
SELECT 
    Category,
    COUNT(DISTINCT Order_ID) AS total_orders,
    SUM(Quantity) AS units_sold,
    ROUND(SUM(Total_Sales_INR), 2) AS category_sales_inr,
    ROUND(SUM(Profit_INR), 2) AS category_profit_inr,
    ROUND((SUM(Profit_INR) / SUM(Total_Sales_INR)) * 100, 2) AS profit_margin_pct,
    DENSE_RANK() OVER (ORDER BY SUM(Total_Sales_INR) DESC) AS sales_rank
FROM amazon_sales
GROUP BY Category
ORDER BY category_sales_inr DESC;
"""
pd.read_sql(query_3, conn)


,Category,total_orders,units_sold,category_sales_inr,category_profit_inr,profit_margin_pct,sales_rank
0,Electronics & Mobiles,3045,7579,1.235695e+08,26310257.24,21.29,1
1,Home & Kitchen,1888,4684,1.797322e+07,3837268.36,21.35,2
2,Apparel & Fashion,2580,6452,9.831025e+06,2085354.00,21.21,3
3,Beauty & Personal Care,1499,3734,2.889178e+06,604390.41,20.92,4
4,Pantry & Groceries,988,2477,1.526964e+06,329738.48,21.59,5


### Query 4: Top 10 Best-Selling Products by Revenue (Slide 3)


In [5]:
query_4 = """
SELECT 
    Product,
    Category,
    SUM(Quantity) AS units_sold,
    ROUND(SUM(Total_Sales_INR), 2) AS total_sales_inr,
    ROUND(SUM(Profit_INR), 2) AS total_profit_inr,
    ROUND((SUM(Profit_INR) / SUM(Total_Sales_INR)) * 100, 2) AS profit_margin_pct
FROM amazon_sales
GROUP BY Product, Category
ORDER BY total_sales_inr DESC
LIMIT 10;
"""
pd.read_sql(query_4, conn)


,Product,Category,units_sold,total_sales_inr,total_profit_inr,profit_margin_pct
0,Laptop Backpack,Electronics & Mobiles,1641,26529187.41,5628644.03,21.22
1,Power Bank 20000mAh,Electronics & Mobiles,1573,25548449.07,5496680.51,21.51
2,5G Smartphone,Electronics & Mobiles,1478,24330766.23,5109132.92,21.00
3,Wireless Earbuds,Electronics & Mobiles,1475,23848640.89,5204191.90,21.82
4,Smartwatch,Electronics & Mobiles,1412,23312463.47,4871607.88,20.90
5,Stainless Steel Dinner Set,Home & Kitchen,953,3865114.25,819333.80,21.20
6,Cotton Bedsheet Double,Home & Kitchen,998,3789133.82,843105.06,22.25
7,Water Purifier,Home & Kitchen,992,3785924.80,794225.82,20.98
8,Induction Cooktop,Home & Kitchen,907,3510999.56,761204.61,21.68
9,Mixer Grinder 750W,Home & Kitchen,834,3022046.50,619399.07,20.50


### Query 5: Order Status Breakdown & Revenue Loss Analysis (Slide 4)
Computes order counts, percentage shares, realized revenue, and lost revenue using conditional aggregation.


In [6]:
query_5 = """
SELECT 
    Order_Status,
    COUNT(*) AS order_count,
    ROUND((COUNT(*) * 100.0) / (SELECT COUNT(*) FROM amazon_sales), 2) AS pct_of_total_orders,
    ROUND(SUM(Total_Sales_INR), 2) AS gross_sales_value_inr,
    ROUND(SUM(Profit_INR), 2) AS realized_profit_inr,
    ROUND(SUM(CASE WHEN Order_Status IN ('Returned', 'Cancelled') THEN Total_Sales_INR ELSE 0 END), 2) AS lost_revenue_inr
FROM amazon_sales
GROUP BY Order_Status
ORDER BY order_count DESC;
"""
pd.read_sql(query_5, conn)


,Order_Status,order_count,pct_of_total_orders,gross_sales_value_inr,realized_profit_inr,lost_revenue_inr
0,Delivered,8147,81.47,1.258942e+08,29697472.88,0.00
1,Shipped,868,8.68,1.424694e+07,3469535.61,0.00
2,Cancelled,500,5.00,7.688159e+06,0.00,7688159.19
3,Returned,485,4.85,7.960583e+06,0.00,7960583.45


### Query 6: Category-Wise Return & Cancellation Rates (Slide 4)


In [7]:
query_6 = """
SELECT 
    Category,
    COUNT(*) AS total_orders,
    SUM(CASE WHEN Order_Status = 'Returned' THEN 1 ELSE 0 END) AS returned_orders,
    SUM(CASE WHEN Order_Status = 'Cancelled' THEN 1 ELSE 0 END) AS cancelled_orders,
    ROUND((SUM(CASE WHEN Order_Status = 'Returned' THEN 1.0 ELSE 0 END) / COUNT(*)) * 100, 2) AS return_rate_pct,
    ROUND((SUM(CASE WHEN Order_Status = 'Cancelled' THEN 1.0 ELSE 0 END) / COUNT(*)) * 100, 2) AS cancellation_rate_pct,
    ROUND(SUM(CASE WHEN Order_Status IN ('Returned', 'Cancelled') THEN Total_Sales_INR ELSE 0 END), 2) AS revenue_lost_inr
FROM amazon_sales
GROUP BY Category
ORDER BY return_rate_pct DESC;
"""
pd.read_sql(query_6, conn)


,Category,total_orders,returned_orders,cancelled_orders,return_rate_pct,cancellation_rate_pct,revenue_lost_inr
0,Electronics & Mobiles,3045,164,136,5.39,4.47,12423842.80
1,Beauty & Personal Care,1499,75,84,5.00,5.60,297818.41
2,Home & Kitchen,1888,90,91,4.77,4.82,1774766.26
3,Pantry & Groceries,988,46,41,4.66,4.15,134738.65
4,Apparel & Fashion,2580,110,148,4.26,5.74,1017576.52


### Query 7: Payment Method Performance (Slide 5)


In [8]:
query_7 = """
SELECT 
    Payment_Method,
    COUNT(DISTINCT Order_ID) AS total_orders,
    ROUND(SUM(Total_Sales_INR), 2) AS total_sales_inr,
    ROUND(SUM(Profit_INR), 2) AS total_profit_inr,
    ROUND(SUM(Total_Sales_INR) / COUNT(DISTINCT Order_ID), 2) AS aov_inr,
    ROUND((SUM(CASE WHEN Order_Status = 'Returned' THEN 1.0 ELSE 0 END) / COUNT(*)) * 100, 2) AS return_rate_pct,
    ROUND((SUM(Total_Sales_INR) * 100.0) / (SELECT SUM(Total_Sales_INR) FROM amazon_sales), 2) AS sales_share_pct
FROM amazon_sales
GROUP BY Payment_Method
ORDER BY total_sales_inr DESC;
"""
pd.read_sql(query_7, conn)


,Payment_Method,total_orders,total_sales_inr,total_profit_inr,aov_inr,return_rate_pct,sales_share_pct
0,UPI,4972,78515436.26,16653494.14,15791.52,4.87,50.40
1,Credit/Debit Card,2074,31069230.90,6525646.17,14980.34,5.45,19.94
2,Cash on Delivery (COD),1458,23101583.64,5008107.37,15844.71,3.64,14.83
3,Net Banking,1024,15419486.46,3292296.82,15058.09,6.05,9.90
4,Amazon Pay Later,472,7684156.63,1687463.99,16279.99,3.18,4.93


### Query 8: Fulfillment Channel Efficiency (Slide 5)


In [9]:
query_8 = """
SELECT 
    Fulfillment,
    COUNT(DISTINCT Order_ID) AS total_orders,
    ROUND(SUM(Total_Sales_INR), 2) AS total_sales_inr,
    ROUND(SUM(Profit_INR), 2) AS total_profit_inr,
    ROUND((SUM(Profit_INR) / SUM(Total_Sales_INR)) * 100, 2) AS profit_margin_pct,
    ROUND((SUM(CASE WHEN Order_Status = 'Delivered' THEN 1.0 ELSE 0 END) / COUNT(*)) * 100, 2) AS delivery_success_rate_pct,
    ROUND((SUM(CASE WHEN Order_Status = 'Returned' THEN 1.0 ELSE 0 END) / COUNT(*)) * 100, 2) AS return_rate_pct
FROM amazon_sales
GROUP BY Fulfillment
ORDER BY total_sales_inr DESC;
"""
pd.read_sql(query_8, conn)


,Fulfillment,total_orders,total_sales_inr,total_profit_inr,profit_margin_pct,delivery_success_rate_pct,return_rate_pct
0,Amazon (FBA),7054,1.075283e+08,22882414.52,21.28,82.01,4.76
1,Seller Flex,1966,3.119839e+07,6815054.52,21.84,81.08,4.93
2,Merchant (FBM),980,1.706321e+07,3469539.45,20.33,78.37,5.31


### Query 9: Geographic Performance by State (Slide 5)


In [10]:
query_9 = """
SELECT 
    Ship_State,
    COUNT(DISTINCT Order_ID) AS total_orders,
    SUM(Quantity) AS units_sold,
    ROUND(SUM(Total_Sales_INR), 2) AS total_sales_inr,
    ROUND(SUM(Profit_INR), 2) AS total_profit_inr,
    ROUND(SUM(Total_Sales_INR) / COUNT(DISTINCT Order_ID), 2) AS aov_inr,
    ROUND((SUM(Profit_INR) / SUM(Total_Sales_INR)) * 100, 2) AS profit_margin_pct,
    ROUND((SUM(CASE WHEN Order_Status = 'Returned' THEN 1.0 ELSE 0 END) / COUNT(*)) * 100, 2) AS state_return_rate_pct
FROM amazon_sales
GROUP BY Ship_State
ORDER BY total_sales_inr DESC;
"""
pd.read_sql(query_9, conn)


,Ship_State,total_orders,units_sold,total_sales_inr,total_profit_inr,aov_inr,profit_margin_pct,state_return_rate_pct
0,Maharashtra,1840,4585,29554237.21,6301731.56,16062.09,21.32,5.05
1,Karnataka,1515,3765,24415704.16,5273235.35,16115.98,21.60,4.82
2,Delhi,1444,3610,22361687.04,4699095.38,15485.93,21.01,4.92
3,Tamil Nadu,1109,2785,17889855.25,3856864.11,16131.52,21.56,4.87
4,Uttar Pradesh,991,2486,14140675.36,2978266.81,14269.10,21.06,4.74
5,Telangana,906,2272,13961458.53,2944127.16,15410.00,21.09,5.74
6,West Bengal,665,1645,10864550.51,2344788.08,16337.67,21.58,4.21
7,Gujarat,591,1484,9369304.18,2008021.80,15853.31,21.43,4.06
8,Rajasthan,456,1144,6620697.25,1383028.68,14519.07,20.89,5.48
9,Kerala,483,1150,6611724.40,1377849.56,13688.87,20.84,3.73


In [11]:
conn.close()
print("SQL Analysis queries completed successfully.")


SQL Analysis queries completed successfully.
